# 03 - Exploratory Data Analysis

**Owner:** Analysis Lead  
**Goal:** Convert the cleaned hospital operations data into decision-ready KPI tables, charts, and insight candidates for the report, dashboard, and recommendations.

This notebook is intentionally business-facing: every table should answer a question about cost, outcomes, access, or operational quality.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ANALYSIS_DIR = PROCESSED_DIR / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
    sns.set_theme(style='whitegrid', palette='Set2')
except Exception as exc:
    HAS_PLOTS = False
    print('Plot libraries unavailable. Tables will still run. Install matplotlib and seaborn for charts.')
    print(type(exc).__name__, exc)

In [ ]:
cleaned_paths = sorted(PROCESSED_DIR.glob('cleaned_hospital_*.csv'))
assert cleaned_paths, 'No cleaned hospital files found in data/processed/'

frames = []
for path in cleaned_paths:
    part = pd.read_csv(path)
    part['Source_File'] = path.name
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(cleaned_paths)} cleaned files')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
display(df.head())

## 1. Data Quality Snapshot

Before analysis, confirm the cleaned data is analysis-ready: no missing values, no duplicate rows, expected column types, and enough rows for group-level analysis.

In [ ]:
quality_summary = pd.DataFrame({
    'metric': ['rows', 'columns', 'missing_values', 'duplicate_rows', 'hospitals', 'diagnoses', 'regions'],
    'value': [
        len(df),
        df.shape[1],
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        df['Hospital'].nunique(),
        df['Diagnosis'].nunique(),
        df['Region'].nunique(),
    ],
})

display(quality_summary)
display(df.dtypes.rename('dtype').reset_index().rename(columns={'index': 'column'}))

In [ ]:
categorical_cols = ['Hospital', 'Hospital_Type', 'Diagnosis', 'Gender', 'Age_Group', 'Cost_Tier', 'Economic_Status', 'Outcome', 'Insurance', 'Region']
cardinality = []
for col in categorical_cols:
    if col in df.columns:
        cardinality.append({
            'column': col,
            'unique_values': df[col].nunique(dropna=False),
            'top_value': df[col].mode().iloc[0],
            'top_count': int(df[col].value_counts(dropna=False).iloc[0]),
        })
cardinality = pd.DataFrame(cardinality)
display(cardinality)

## 2. KPI Framework

Use these KPIs consistently in the Tableau dashboard and final report.

- **Patient Volume:** count of records/patients.
- **Average Treatment Cost:** mean cost burden.
- **Median Treatment Cost:** robust cost indicator when outliers exist.
- **Recovery Rate:** recovered patients divided by total patients.
- **Mortality Rate:** deceased patients divided by total patients.
- **Under-Treatment Rate:** patients still under treatment divided by total patients.
- **Insurance Coverage Rate:** insured patients divided by total patients.
- **Operational Scores:** average doctor availability, cleanliness score, and doctor experience.

In [ ]:
def kpi_table(data, group_cols):
    grouped = data.groupby(group_cols, dropna=False)
    out = grouped.agg(
        Patient_Volume=('Patient_ID', 'count'),
        Avg_Treatment_Cost=('Treatment_Cost', 'mean'),
        Median_Treatment_Cost=('Treatment_Cost', 'median'),
        Avg_Doctor_Experience=('Doctor_Experience_Years', 'mean'),
        Avg_Doctor_Availability=('Doctor_Availability', 'mean'),
        Avg_Cleanliness_Score=('Cleanliness_Score', 'mean'),
        Insurance_Coverage_Rate=('Insurance', lambda s: (s == 'Yes').mean()),
        Recovery_Rate=('Outcome', lambda s: (s == 'Recovered').mean()),
        Mortality_Rate=('Outcome', lambda s: (s == 'Deceased').mean()),
        Under_Treatment_Rate=('Outcome', lambda s: (s == 'Under Treatment').mean()),
    ).reset_index()

    rate_cols = ['Insurance_Coverage_Rate', 'Recovery_Rate', 'Mortality_Rate', 'Under_Treatment_Rate']
    out[rate_cols] = (out[rate_cols] * 100).round(2)
    numeric_cols = out.select_dtypes(include='number').columns
    out[numeric_cols] = out[numeric_cols].round(2)
    return out.sort_values(['Patient_Volume'], ascending=False)

hospital_kpis = kpi_table(df, ['Hospital', 'Hospital_Type'])
diagnosis_kpis = kpi_table(df, ['Diagnosis'])
region_kpis = kpi_table(df, ['Region'])
age_kpis = kpi_table(df, ['Age_Group'])
economic_kpis = kpi_table(df, ['Economic_Status'])
insurance_kpis = kpi_table(df, ['Insurance'])

for name, table in {
    'hospital_kpis': hospital_kpis,
    'diagnosis_kpis': diagnosis_kpis,
    'region_kpis': region_kpis,
    'age_group_kpis': age_kpis,
    'economic_status_kpis': economic_kpis,
    'insurance_kpis': insurance_kpis,
}.items():
    table.to_csv(ANALYSIS_DIR / f'{name}.csv', index=False)

print(f'Saved KPI tables to {ANALYSIS_DIR}')
display(hospital_kpis)
display(diagnosis_kpis)

## 3. Cost Analysis

Primary question: which hospital, diagnosis, patient segment, or operational factor is associated with higher treatment cost?

In [ ]:
cost_by_hospital = df.groupby('Hospital')['Treatment_Cost'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2).sort_values('mean', ascending=False)
cost_by_diagnosis = df.groupby('Diagnosis')['Treatment_Cost'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2).sort_values('mean', ascending=False)
cost_by_age = df.groupby('Age_Group')['Treatment_Cost'].agg(['count', 'mean', 'median', 'std']).round(2).sort_values('mean', ascending=False)
cost_by_insurance = df.groupby('Insurance')['Treatment_Cost'].agg(['count', 'mean', 'median', 'std']).round(2).sort_values('mean', ascending=False)

display(cost_by_hospital)
display(cost_by_diagnosis)
display(cost_by_age)
display(cost_by_insurance)

In [ ]:
if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.boxplot(data=df, x='Hospital', y='Treatment_Cost', ax=axes[0])
    axes[0].set_title('Treatment Cost Distribution by Hospital')
    axes[0].tick_params(axis='x', rotation=20)

    sns.boxplot(data=df, x='Diagnosis', y='Treatment_Cost', ax=axes[1])
    axes[1].set_title('Treatment Cost Distribution by Diagnosis')
    axes[1].tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print('Install matplotlib and seaborn to render boxplots.')

## 4. Outcome Analysis

Primary question: where are recovery, mortality, and under-treatment rates strongest or weakest?

In [ ]:
outcome_by_hospital = pd.crosstab(df['Hospital'], df['Outcome'], normalize='index').mul(100).round(2)
outcome_by_diagnosis = pd.crosstab(df['Diagnosis'], df['Outcome'], normalize='index').mul(100).round(2)
outcome_by_age = pd.crosstab(df['Age_Group'], df['Outcome'], normalize='index').mul(100).round(2)
outcome_by_economic = pd.crosstab(df['Economic_Status'], df['Outcome'], normalize='index').mul(100).round(2)

display(outcome_by_hospital)
display(outcome_by_diagnosis)
display(outcome_by_age)
display(outcome_by_economic)

outcome_by_hospital.to_csv(ANALYSIS_DIR / 'outcome_mix_by_hospital.csv')
outcome_by_diagnosis.to_csv(ANALYSIS_DIR / 'outcome_mix_by_diagnosis.csv')

In [ ]:
if HAS_PLOTS:
    ax = outcome_by_diagnosis[['Recovered', 'Under Treatment', 'Deceased']].plot(kind='bar', stacked=True, figsize=(10, 5))
    ax.set_title('Outcome Mix by Diagnosis')
    ax.set_ylabel('Share of Patients (%)')
    ax.legend(title='Outcome', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print('Install matplotlib to render stacked bar charts.')

## 5. Operational Factor Analysis

Primary question: do availability, cleanliness, and experience move with cost or patient outcomes?

In [ ]:
numeric_cols = ['Age', 'Treatment_Cost', 'Doctor_Experience_Years', 'Doctor_Availability', 'Cleanliness_Score']
corr = df[numeric_cols].corr(method='spearman').round(3)
display(corr)
corr.to_csv(ANALYSIS_DIR / 'spearman_correlation_matrix.csv')

ops_by_hospital = df.groupby('Hospital').agg(
    Avg_Doctor_Availability=('Doctor_Availability', 'mean'),
    Avg_Cleanliness_Score=('Cleanliness_Score', 'mean'),
    Avg_Doctor_Experience=('Doctor_Experience_Years', 'mean'),
    Avg_Treatment_Cost=('Treatment_Cost', 'mean'),
    Recovery_Rate=('Outcome', lambda s: (s == 'Recovered').mean() * 100),
).round(2).sort_values('Avg_Treatment_Cost', ascending=False)
display(ops_by_hospital)

In [ ]:
if HAS_PLOTS:
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0)
    plt.title('Spearman Correlation Matrix')
    plt.tight_layout()
    plt.show()
else:
    print('Install matplotlib and seaborn to render the correlation heatmap.')

## 6. Insight Candidates for Report

These are not final recommendations yet. Treat them as evidence-backed leads that the Strategy Lead can convert into recommendations after mentor/team review.

In [ ]:
overall_avg_cost = df['Treatment_Cost'].mean()
overall_recovery = (df['Outcome'].eq('Recovered').mean() * 100)
overall_mortality = (df['Outcome'].eq('Deceased').mean() * 100)

highest_cost_hospital = hospital_kpis.sort_values('Avg_Treatment_Cost', ascending=False).iloc[0]
lowest_cost_hospital = hospital_kpis.sort_values('Avg_Treatment_Cost', ascending=True).iloc[0]
highest_cost_diagnosis = diagnosis_kpis.sort_values('Avg_Treatment_Cost', ascending=False).iloc[0]
lowest_recovery_diagnosis = diagnosis_kpis.sort_values('Recovery_Rate', ascending=True).iloc[0]
highest_under_treatment = diagnosis_kpis.sort_values('Under_Treatment_Rate', ascending=False).iloc[0]

insight_candidates = pd.DataFrame([
    {
        'theme': 'Cost variation by hospital',
        'evidence': f"{highest_cost_hospital['Hospital']} has the highest average treatment cost ({highest_cost_hospital['Avg_Treatment_Cost']:.2f}), while {lowest_cost_hospital['Hospital']} is lowest ({lowest_cost_hospital['Avg_Treatment_Cost']:.2f}).",
        'decision_use': 'Investigate whether the cost gap is justified by case mix, private/government type, or process inefficiency.'
    },
    {
        'theme': 'Diagnosis-level cost burden',
        'evidence': f"{highest_cost_diagnosis['Diagnosis']} is the highest-cost diagnosis with average cost {highest_cost_diagnosis['Avg_Treatment_Cost']:.2f}, above the overall average {overall_avg_cost:.2f}.",
        'decision_use': 'Prioritise cost-control protocols for high-cost diagnosis categories.'
    },
    {
        'theme': 'Recovery performance',
        'evidence': f"{lowest_recovery_diagnosis['Diagnosis']} has the lowest recovery rate ({lowest_recovery_diagnosis['Recovery_Rate']:.2f}%) compared with overall recovery {overall_recovery:.2f}%.",
        'decision_use': 'Review treatment pathways and follow-up capacity for lower-recovery diagnosis groups.'
    },
    {
        'theme': 'Treatment backlog',
        'evidence': f"{highest_under_treatment['Diagnosis']} has the highest under-treatment rate ({highest_under_treatment['Under_Treatment_Rate']:.2f}%).",
        'decision_use': 'Use staffing, bed allocation, or triage changes to reduce active treatment backlog.'
    },
    {
        'theme': 'Outcome baseline',
        'evidence': f"Overall mortality is {overall_mortality:.2f}% across {len(df):,} records.",
        'decision_use': 'Use mortality as a monitored risk KPI, segmented by diagnosis, age group, and hospital.'
    },
])

insight_candidates.to_csv(ANALYSIS_DIR / 'insight_candidates.csv', index=False)
display(insight_candidates)

## 7. Handoff Notes

- Visualization Lead should use `data/processed/analysis/*.csv` for KPI tables and `data/processed/hospital_tableau_ready.csv` after running Notebook 05.
- Strategy Lead should turn insight candidates into 3-5 recommendations only after validating statistical significance in Notebook 04.
- PPT & Quality Lead should use the decision-language insights, not raw chart descriptions.